# CS 6470 A2 — Feature selection + k-NN vs linear

**70 points** · Modules 4–5 (due Module 5) · plan for about 4–6 focused hours

**Before you submit, rename this file to `CS6470_A2_LastName_FirstName.ipynb`.**
Open it in [Google Colab](https://colab.research.google.com/) (File → Upload notebook) or local Jupyter (Python 3.10+).

## How this notebook works

- It has **5 numbered tasks**, top to bottom. Cells marked **`# YOUR CODE`** and Markdown cells marked **YOUR ANSWER** are the graded work. Cells marked *Runs as-is* are provided — run them and read them.
- Every coding task ends with a **Checkpoint** describing what you should see, so you always know whether you are on track. Small differences (±0.01–0.02 on scores) are normal.
- **Hints name the tool** (class, argument, or doc link), not a line to paste. Newer to Python? Follow the Hints. Comfortable? Try each task from the description first. Experienced? There is an ungraded stretch at the end of the assignment page.
- A `# YOUR CODE` cell raises `NotImplementedError` until you replace that line — that is the notebook telling you where to work, not a bug.
- Before submitting: **Runtime → Restart and run all.** Every cell must run with no errors and no `NotImplementedError` left.
- Two cells in this notebook take **a few minutes** each (mutual information on 39k rows; k-NN predictions). That is normal — the notes say where.

**Protocol clarification (September 14):** the held-out 20% is a validation set because Tasks 2–3 use it to choose a model and k. There is no independent final test in A2. A3 introduces CV plus a final test. If you already began with variables named `X_test` and `y_test`, you may keep those names and explain their validation role; the task count, points and deadline are unchanged. The supplied `score_features` helper treats numeric and one-hot columns appropriately.


## Setup: the A1 protocol *(runs as-is — read it, it should look familiar)*

Same table as A1 (**Adult Census Income**), same split, same preprocessing. This cell is provided so your time goes to the new ideas — selection and model comparison — instead of re-typing A1. One deliberate choice: **every column is kept, including `fnlwgt`**. Task 4 asks you what the selector does about that.

After one-hot encoding there are about **105 feature columns** — enough that choosing 20 actually means something.


In [1]:
# Runs as-is — A1 protocol: load, split, preprocessing (all columns kept on purpose).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

adult = fetch_openml("adult", version=2, as_frame=True, parser="auto")
X, y = adult.data, adult.target

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
])
print("train:", X_train.shape, "validation:", X_val.shape)

# Numeric measurements use continuous MI; one-hot indicators are discrete.
# ColumnTransformer outputs numeric columns first, then one-hot columns.
def score_features(X_encoded, y_labels):
    discrete = np.arange(X_encoded.shape[1]) >= len(num_cols)
    return mutual_info_classif(X_encoded, y_labels,
                               discrete_features=discrete, random_state=42)


train: (39073, 14) validation: (9769, 14)


## Task 1 of 5 — Put selection inside the Pipeline

Selection is part of *training* — if it saw the validation rows, the validation score would be flattered. Putting `SelectKBest` between the preprocessing and the model guarantees it never does.

In code:

1. Build and fit `pipe`: `pre` → `SelectKBest(score_features, k=20)` → `LogisticRegression(max_iter=1000)`.
2. Print the validation macro-F1.
3. Print the **names** of the 20 surviving features — names, not indices; nobody can act on "feature 37".

*(The fit takes 1–2 minutes — mutual information looks at every feature/target pair on 39k rows.)*

**Hints.** Tools only — assemble them yourself:
- [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): reuse `pre`, then `SelectKBest(score_features, k=20)`, then `LogisticRegression(max_iter=1000)`.
- Test score: [`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) with `average="macro"`.
- Names: the `pre` step's `get_feature_names_out()` and the selector step's `get_support()` — index one with the other.

**Checkpoint.** Show the requested named features or score table and explain your observed result. Exact scores and winning models can vary; a specific number is not required.


In [2]:
# YOUR CODE — Task 1. Write one small block under each comment.

# 1. pipe: pre -> SelectKBest(score_features, k=20) -> LogisticRegression(max_iter=1000)
pipe = Pipeline([
    ("pre", pre),
    ("select", SelectKBest(score_features, k=20)),
    ("clf", LogisticRegression(max_iter=1000)),
])

# 2. Fit on the training split. Print validation macro-F1.
pipe.fit(X_train, y_train)

print("Macro F1 score on validation set:", 
      f1_score(y_val, pipe.predict(X_val), average="macro"))

# 3. Print the 20 surviving feature names (names, not indices).
print(pipe.named_steps["pre"].get_feature_names_out()[pipe.named_steps["select"].get_support()])


Macro F1 score on validation set: 0.771918027260458
['num__age' 'num__fnlwgt' 'num__education-num' 'num__capital-gain'
 'num__capital-loss' 'num__hours-per-week' 'cat__education_Bachelors'
 'cat__education_Masters' 'cat__education_Prof-school'
 'cat__marital-status_Divorced' 'cat__marital-status_Married-civ-spouse'
 'cat__marital-status_Never-married' 'cat__occupation_Exec-managerial'
 'cat__occupation_Other-service' 'cat__relationship_Husband'
 'cat__relationship_Not-in-family' 'cat__relationship_Own-child'
 'cat__relationship_Unmarried' 'cat__sex_Female' 'cat__sex_Male']


## Task 2 of 5 — Fair three-way comparison

One helper, three models, identical protocol — that is what makes a comparison fair. Complete `evaluate` below and run it for the dummy, k-NN, and logistic regression.

**Hints.** Inside `evaluate`: same three-step Pipeline as Task 1, but the last step is the `clf` argument. Fit on train, print `name` plus validation macro-F1, return the fitted pipeline. *(Each call takes ~1–2 minutes.)*

**Checkpoint.** Show the requested named features or score table and explain your observed result. Exact scores and winning models can vary; a specific number is not required.


In [3]:
# YOUR CODE — Task 2. Complete the numbered blanks inside evaluate().
def evaluate(name, clf):
    """Fit pre -> SelectKBest(score_features, k=20) -> clf on the training
    split, print `name` and the validation macro-F1, and return the fitted pipeline."""
    # 1. Build the pipeline (reuse pre; last step is clf)
    pipe = Pipeline([
        ("pre", pre),
        ("select", SelectKBest(score_features, k=20)),
        ("clf", clf),
    ])

    # 2. Fit on the training split
    pipe.fit(X_train, y_train)

    # 3. Print name + validation macro-F1; return the fitted pipeline

    print(name, f1_score(y_val, pipe.predict(X_val), average="macro"))
    return pipe

for name, clf in [
    ("dummy", DummyClassifier(strategy="most_frequent")),
    ("k-NN (k=7)", KNeighborsClassifier(n_neighbors=7)),
    ("logistic", LogisticRegression(max_iter=1000)),
]:
    evaluate(name, clf)


dummy 0.43203488372093024
k-NN (k=7) 0.7621178725333766
logistic 0.771918027260458


## Task 3 of 5 — k-NN's bias–variance dial

k is the memorization dial: k=1 means "copy the label of the single closest row"; large k means "average over a whole neighborhood".

In code: for `k` in `(1, 7, 25)`, fit the same pipeline with `KNeighborsClassifier(n_neighbors=k)` and print **train** macro-F1 and **validation** macro-F1 side by side. *(Scoring the training split makes k-NN predict 39k rows — this cell is the slow one, a few minutes total.)*

**Hints.** Loop `k` in `(1, 7, 25)`. Each time: `pre` → `SelectKBest(score_features, k=20)` → `KNeighborsClassifier(n_neighbors=k)`. Print train *and* validation macro-F1.

**Checkpoint.** Show the requested named features or score table and explain your observed result. Exact scores and winning models can vary; a specific number is not required.


In [4]:
# YOUR CODE — Task 3. Write one small block under each comment.

# 1. For k in (1, 7, 25): build the pipeline, fit on train
for k in (1, 7, 25):
    pipe = Pipeline([
        ("pre", pre),
        ("select", SelectKBest(score_features, k=20)),
        ("clf", KNeighborsClassifier(n_neighbors=k)),
    ])

    pipe.fit(X_train, y_train)

# 2. Print k, train macro-F1, and validation macro-F1
    print(k, f1_score(y_train, pipe.predict(X_train), average="macro")
          , f1_score(y_val, pipe.predict(X_val), average="macro"))


1 0.9992619529329319 0.733210029411715
7 0.8184645907037003 0.7621178725333766
25 0.7906567114135152 0.7733803574624624


### Task 3 (written) — read the gap — YOUR ANSWER

**TODO (2–4 sentences): Which k memorizes? How does the train-vs-validation gap show it? Which k would you actually ship, and why?**

K=1 memorizes the training data and is overfitting because its training F1 is nearly perfect, while its validation F1 is much lower. I would actually ship k=25 because it had the highest validation F1 and the smallest gap between the training and validation scores, showing that it generalized better and was less likely to overfit.

## Task 4 of 5 — Selection questions — YOUR ANSWER

Look at your Task 1 survivor list.

**Q1.** Name three survivors that make real-world sense for an income model, in one sentence each.

**TODO:** 
`num_age` because income relates to career and life experience, someone younger could be in college and compared to someone older they could have a career. 

`num_edu` because education influences what types of jobs someone qualifies for. 

`num_hours` because how many hours someone works a week can relate to how much they make, especially an hourly employee.

**Q2.** Did `num__fnlwgt` survive? A1 established it is a sampling artifact, not a fact about the person. What does the selector's verdict (either way) tell you about automated selection vs domain knowledge?

**TODO:** 
`num_fnlwgt` survived, this shows me that the feature selector doesn't understand the meaning of the columns. It looks at the statistical relationship with our target. It also reiterated that automated feature selection isn't the same as human judgement, so having a human in the loop and domain knowledge is still important.

**Q3.** Are any survivors a fairness worry if this model priced a product or screened applications? (Look for `sex`, `marital-status`, `relationship`.) One or two sentences.

**TODO:** 
Yes, there's fairness worries since sex, marital status, and relationship survived. Even if these features have a statistical relationship with income, using them to screen applications or price a product could introduce bias and treat people differently based on those characteristics.


## Task 5 of 5 — Memo (≤1 page) — YOUR ANSWER

For a non-CS manager. Replace each **TODO**.

### What feature selection bought us

**TODO:** Feature selection gave me a more focused model with fewer inputs. My input went from roughly 105 down to 20 and the performance changed from 0.7818 before the selection to 0.7719 after, so my performance didn't go down that much. This means there are fewer things to collect, store, and monitor in the future.

### Which model I would use

**TODO:** For this dataset, I would use a k-NN model. It was close behind logistic at first with a validation score of 0.7621, but after I changed it from k=7 to k=25, it went to 0.7734, which is better. It also had the smallest validation gap, so it is more consistent and generalizes better.

### When k-NN is the wrong tool

**TODO:** k-NN is the wrong tool when you need stable coefficients, faster speeds, or a simple linear boundary. Companies that need to compare less data at a time could use k-NN more often than companies that need to compare larger datasets. If a large company needs millions of predictions, k-NN could be slower because it has to find its neighbors, calculate distances, and then compare those records. That could make k-NN a worse choice if the company needs something simple and fast instead.

### AI disclosure

**TODO:** I used ChatGPT as a tutor to help me understand how everything was working together. All the code was mine, but it helped me troubleshoot syntax errors within my code.


## Before you submit

- [ ] File renamed to `CS6470_A2_LastName_FirstName.ipynb`
- [ ] **Runtime → Restart and run all** — every cell ran, no errors, no `NotImplementedError` anywhere
- [ ] All 5 tasks done: every `# YOUR CODE` cell has your code, every **YOUR ANSWER** / **TODO** prompt has your words
- [ ] Memo written in the Markdown cells above (it stays inside this notebook — no separate PDF)
- [ ] AI-use disclosure filled in

Upload the `.ipynb` file to this assignment on Canvas (in Colab: File → Download → Download .ipynb).

Integrity: the notebook and memo must be yours. AI may help explain an error; it may not be the submitted analysis.
